In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# All HF downloads will read/write here
DRIVE_ROOT = "/content/drive/MyDrive/msc-deepfake"
HF_CACHE = f"{DRIVE_ROOT}/hf_cache"

os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE

os.makedirs(HF_CACHE, exist_ok=True)

# Verify
!ls -la /content/drive/MyDrive/msc-deepfake/
print(f"\nHF_HOME set to: {os.environ['HF_HOME']}")

total 16
drwx------ 2 root root 4096 Jul 19 08:40 batch_prompts
drwx------ 2 root root 4096 Jul 19 08:40 generated_videos
drwx------ 2 root root 4096 Jul 19 08:40 hf_cache
drwx------ 2 root root 4096 Jul 19 08:41 notebooks

HF_HOME set to: /content/drive/MyDrive/msc-deepfake/hf_cache


In [3]:
BATCH_PROMPTS = """# Week 2 batch prompts — v1
# Format: prompt_id|category|prompt_text
# Categories: portrait, hands, multi_person, motion, text_scene, texture, animal, edge_case

# --- PORTRAITS (facial coherence stress) ---
w2_001|portrait|A woman in her thirties smiling and speaking directly to the camera in a park, natural daylight
w2_002|portrait|An older man laughing while telling a story, cafe interior, warm lighting
w2_003|portrait|A young man with glasses reading a book, close-up on his face, soft afternoon light
w2_004|portrait|A woman with curly hair looking thoughtfully out a rainy window, muted colours
w2_005|portrait|A bearded chef tasting food from a spoon, kitchen background, dramatic lighting

# --- HANDS AND FINE MOTOR (semantic coherence stress) ---
w2_006|hands|Close-up of a person typing on a laptop keyboard, both hands visible, coffee shop background
w2_007|hands|A hand pouring milk from a glass bottle into a coffee cup, kitchen counter
w2_008|hands|Someone tying their shoelaces, close-up on hands and shoes, park bench
w2_009|hands|A person writing in a notebook with a fountain pen, close-up on the hand and paper
w2_010|hands|Two hands shuffling a deck of playing cards on a wooden table

# --- MULTI-PERSON INTERACTION (identity + spatial coherence) ---
w2_011|multi_person|Two friends walking down a busy street talking to each other, afternoon light
w2_012|multi_person|A family of four eating dinner at a dining table, warm evening light
w2_013|multi_person|Three colleagues in a meeting room having a discussion, whiteboard visible
w2_014|multi_person|A parent teaching a child to ride a bicycle in a park, sunny day
w2_015|multi_person|A couple sitting on a park bench feeding pigeons, autumn leaves

# --- MOTION AND PHYSICS (temporal consistency stress) ---
w2_016|motion|A person kicking a soccer ball toward the camera, grass field, dynamic motion
w2_017|motion|A jogger running along a beach at sunrise, waves in the background
w2_018|motion|A skateboarder performing a trick on a ramp, urban skate park
w2_019|motion|Water splashing as a swimmer dives into a pool, high-speed capture
w2_020|motion|A tennis player serving a ball on a clay court, dust visible

# --- TEXT IN SCENE (semantic coherence — text handling) ---
w2_021|text_scene|A shopkeeper standing in front of a bookstore, storefront signage visible
w2_022|text_scene|A newspaper vendor at a street corner with newspapers displayed
w2_023|text_scene|A person walking past a chalkboard menu outside a cafe
w2_024|text_scene|A traveller checking a departure board at a train station
w2_025|text_scene|A student writing on a whiteboard in a classroom, equations visible

# --- COMPLEX TEXTURES (texture and detail stress) ---
w2_026|texture|Close-up of a chef chopping vegetables on a wooden cutting board, natural kitchen lighting
w2_027|texture|A weaver working on a traditional loom, colourful threads visible
w2_028|texture|Rain falling on a window with a blurred city view outside, moody atmosphere
w2_029|texture|A potter shaping wet clay on a spinning wheel, close-up on hands and clay
w2_030|texture|A barista pouring latte art into a cup, close-up on the milk and foam

# --- ANIMALS (non-human texture/coherence) ---
w2_031|animal|A golden retriever running in slow motion across a lawn, sunny day
w2_032|animal|A cat stretching lazily on a sunlit windowsill, indoor scene
w2_033|animal|A horse galloping through an open field, wind blowing its mane
w2_034|animal|A parrot preening its feathers on a wooden perch, tropical background
w2_035|animal|A school of fish swimming through a coral reef, underwater scene

# --- EDGE CASES (challenge scenarios) ---
w2_036|edge_case|A magician performing a card trick, hands and cards in mid-motion, stage lighting
w2_037|edge_case|A dancer spinning in a red dress under a spotlight, dramatic shadows
w2_038|edge_case|A person applying makeup in front of a mirror, close-up showing both real face and reflection
w2_039|edge_case|A crowded market at night with many people walking, string lights above
w2_040|edge_case|A person walking their dog past a shop window, both dog and reflection visible
"""

PROMPTS_FILE = f"{DRIVE_ROOT}/batch_prompts/week2_prompts_v1.txt"
os.makedirs(os.path.dirname(PROMPTS_FILE), exist_ok=True)

with open(PROMPTS_FILE, "w") as f:
    f.write(BATCH_PROMPTS)

print(f"Saved {PROMPTS_FILE}")

# Verify
with open(PROMPTS_FILE) as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith("#")]
print(f"\n{len(lines)} prompts loaded")
print("First 3:")
for line in lines[:3]:
    print(f"  {line}")

Saved /content/drive/MyDrive/msc-deepfake/batch_prompts/week2_prompts_v1.txt

40 prompts loaded
First 3:
  w2_001|portrait|A woman in her thirties smiling and speaking directly to the camera in a park, natural daylight
  w2_002|portrait|An older man laughing while telling a story, cafe interior, warm lighting
  w2_003|portrait|A young man with glasses reading a book, close-up on his face, soft afternoon light


In [4]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece
!pip install -q imageio imageio-ffmpeg
print("Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.9 MB/s eta 0:00:00
Installed.


In [5]:
import torch
from diffusers import LTXPipeline

MODEL_ID = "Lightricks/LTX-Video"

print("Loading model — first run will download ~25GB to Drive, subsequent runs load from Drive...")
pipe = LTXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")
print("Model loaded.")
print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading model — first run will download ~25GB to Drive, subsequent runs load from Drive...


model_index.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.
GPU free: 8.79 GB


In [7]:
import datetime
from diffusers.utils import export_to_video

def generate_and_save(prompt_id, category, prompt_text, out_folder):
    """Generate one video and save with structured filename."""
    print(f"\n[{prompt_id}] {category}: {prompt_text[:60]}...")
    video = pipe(
        prompt=prompt_text,
        num_frames=97,
        height=512,
        width=768,
        num_inference_steps=40,
    ).frames[0]

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prompt_id}_ltx_{timestamp}.mp4"
    out_path = f"{out_folder}/{filename}"

    export_to_video(video, out_path, fps=24)
    print(f"Saved: {out_path}")
    return out_path

OUT_FOLDER = f"{DRIVE_ROOT}/generated_videos/ltx"
os.makedirs(OUT_FOLDER, exist_ok=True)
print("Function defined, output folder ready.")

Function defined, output folder ready.


In [8]:
# Loading the first 5 prompts from my batch file
PROMPTS_FILE = f"{DRIVE_ROOT}/batch_prompts/week2_prompts_v1.txt"
with open(PROMPTS_FILE) as f:
    lines = [line.strip() for line in f
             if line.strip()
             and not line.strip().startswith("#")
             and "|" in line]  # must have the pipe separator

first_batch = lines[:5]
print(f"Loaded {len(lines)} total prompts, running first {len(first_batch)}:")
for line in first_batch:
    print(f"  {line}")

# Generate each
for line in first_batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        print(f"Skipping malformed line: {line}")
        continue
    prompt_id, category, prompt_text = parts
    try:
        generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")

Loaded 40 total prompts, running first 5:
  w2_001|portrait|A woman in her twenties with red hair tucked behind her ear, singing softly with her eyes closed, dim indoor stage lighting
  w2_002|portrait|An older man laughing while telling a story, cafe interior, warm lighting
  w2_003|portrait|A young man with glasses reading a book, close-up on his face, soft afternoon light
  w2_004|portrait|A woman with curly hair looking thoughtfully out a rainy window, muted colours
  w2_005|portrait|A bearded chef tasting food from a spoon, kitchen background, dramatic lighting

[w2_001] portrait: A woman in her twenties with red hair tucked behind her ear,...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_001_ltx_20260719_090220.mp4

[w2_002] portrait: An older man laughing while telling a story, cafe interior, ...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_002_ltx_20260719_090321.mp4

[w2_003] portrait: A young man with glasses reading a book, close-up on his fac...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_003_ltx_20260719_090428.mp4

[w2_004] portrait: A woman with curly hair looking thoughtfully out a rainy win...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_004_ltx_20260719_090529.mp4

[w2_005] portrait: A bearded chef tasting food from a spoon, kitchen background...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_005_ltx_20260719_090629.mp4


In [9]:
# Next batch — prompts 6 through 15 (hands + multi-person categories)
next_batch = lines[5:15]

print(f"Running next batch of {len(next_batch)} prompts:")
for line in next_batch:
    print(f"  {line}")

print("\n" + "=" * 60)
print("Starting generation...")
print("=" * 60)

successes = []
failures = []

for line in next_batch:
    parts = line.split("|", 2)
    if len(parts) != 3:
        print(f"Skipping malformed: {line}")
        continue
    prompt_id, category, prompt_text = parts
    try:
        path = generate_and_save(prompt_id, category, prompt_text, OUT_FOLDER)
        successes.append(prompt_id)
    except Exception as e:
        print(f"FAILED on {prompt_id}: {e}")
        failures.append((prompt_id, str(e)))

print("\n" + "=" * 60)
print(f"Batch complete. Success: {len(successes)}, Failed: {len(failures)}")
print(f"Successes: {successes}")
if failures:
    print(f"Failures: {failures}")
print("=" * 60)

Running next batch of 10 prompts:
  w2_006|hands|Close-up of a person typing on a laptop keyboard, both hands visible, coffee shop background
  w2_007|hands|A hand pouring milk from a glass bottle into a coffee cup, kitchen counter
  w2_008|hands|Someone tying their shoelaces, close-up on hands and shoes, park bench
  w2_009|hands|A person writing in a notebook with a fountain pen, close-up on the hand and paper
  w2_010|hands|Two hands shuffling a deck of playing cards on a wooden table
  w2_011|multi_person|Two friends walking down a busy street talking to each other, afternoon light
  w2_012|multi_person|A family of four eating dinner at a dining table, warm evening light
  w2_013|multi_person|Three colleagues in a meeting room having a discussion, whiteboard visible
  w2_014|multi_person|A parent teaching a child to ride a bicycle in a park, sunny day
  w2_015|multi_person|A couple sitting on a park bench feeding pigeons, autumn leaves

Starting generation...

[w2_006] hands: Close

  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_006_ltx_20260719_091137.mp4

[w2_007] hands: A hand pouring milk from a glass bottle into a coffee cup, k...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_007_ltx_20260719_091235.mp4

[w2_008] hands: Someone tying their shoelaces, close-up on hands and shoes, ...


...

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_013_ltx_20260719_091829.mp4

[w2_014] multi_person: A parent teaching a child to ride a bicycle in a park, sunny...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_014_ltx_20260719_091929.mp4

[w2_015] multi_person: A couple sitting on a park bench feeding pigeons, autumn lea...


  0%|          | 0/40 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/msc-deepfake/generated_videos/ltx/w2_015_ltx_20260719_092029.mp4

Batch complete. Success: 10, Failed: 0
Successes: ['w2_006', 'w2_007', 'w2_008', 'w2_009', 'w2_010', 'w2_011', 'w2_012', 'w2_013', 'w2_014', 'w2_015']
